# 09 — `PageMantle-Hard`: Official-8-Calibrated 56-Task Multi-Channel Benchmark & Online DB-CSE Evolution

**Strictly Non-Interfering Standalone Extension**: Writes exclusively to `$DRIVE_ROOT/datasets/pagemantle_hard/`, `$DRIVE_ROOT/runs/pagemantle_hard/`, and new filenames in `$DRIVE_ROOT/paper_artifacts/` (`tab_pagemantle_hard.tex`, `fig_pagemantle_hard.pdf`). Does **NOT** overwrite or modify any output from Notebooks `00`–`08`.

**Why `PageMantle-Hard` Exists (Direct Response to Notebook 08 Audit)**:
Notebook `08` showed that Deepnote's real Official-8 is substantially harder than legacy single-thread `PageMantle-Ext`. `PageMantle-Hard` scales Official-8's exact difficulty profile to **56 DB-grounded multi-channel tasks** (`32` held-out eval + `24` 3-wave online stream, 0% entity overlap):
1. **Multi-Channel Decoy Threads**: Every task interleaves 2 realistic already-handled decoy threads in other channels inside a 10-minute Slack snapshot.
2. **Cross-Service Hidden Silence Traps**: Silence traps look urgent/open on Slack—ONLY querying Linear/Notion/parent Slack records reveals completed owner sign-off (breaking the fake `100% SilAcc` ceiling for lazy baselines!).
3. **Mixed-Intent Guard & Multi-Hop Verification**: Guard tasks demand BOTH safe aggregate metrics AND PII boundary enforcement; action tasks include staging-vs-production, billing sequence, completed-vs-open ticket, and calendar-exception traps.
4. **Strict Deepnote-Faithful Zero-Condition Judge (`DeepnoteOfficialJudge`)**: Exact 0–10 integer anchors + `important_mistakes` zero-conditions + Slack-write-only delivery.
5. **Non-Saturated Online DB-CSE Evolution**: Runs `DBCSEUpdater` (`EscrowGate` + anti-overfitting guardrails + $O(1)$ prompt cap) across 3 waves (`pm_hard_stream_24`) and re-evaluates on untouched `pm_hard_eval_32`.

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/pagemantle_hard",
    "caches",
    "runs/pagemantle_hard",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Multi-Provider Secrets & Configs (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import (
    describe_secret_sources,
    load_autodl_api_key,
    load_jev_api_key,
    load_openai_api_key,
)
from dial_skill.config import load_dial_skill_config, load_models_config

print(
    "Secret lookup diagnostics:",
    json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT, include_autodl=True), indent=2),
)
OPENAI_API_KEY = load_openai_api_key(drive_root=DRIVE_ROOT, required=True)
JEV_API_KEY = load_jev_api_key(drive_root=DRIVE_ROOT, required=True)
AUTODL_API_KEY = load_autodl_api_key(drive_root=DRIVE_ROOT, required=False)
AUTODL_BASE_URL = os.environ.get("AUTODL_BASE_URL", "https://www.autodl.art/api/v1")

models_cfg = load_models_config()
base_cfg = load_dial_skill_config()
print(f"OpenAI key loaded: {bool(OPENAI_API_KEY)} | Jev key loaded: {bool(JEV_API_KEY)} | AutoDL key loaded: {bool(AUTODL_API_KEY)}")


## Step 1 — Build & Verify the 56-Task DB-Grounded `PageMantle-Hard` Suite

Mines verified real records from the extracted 11-service PageMantle dataset, builds `pm_hard_eval_32` (22 action + 10 hard silence traps) and `pm_hard_stream_24` (3 waves × 8 = 16 action + 8 hard silence traps), verifies 0% record-ID overlap (`verify_hard_split_isolation`), and runs a zero-cost `PageMantleHardSession` smoke test.

In [ ]:
# === CELL 3: Build/Load PageMantle-Hard Suite & Zero-Cost Session Smoke Test ===
import pandas as pd
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.benchmarks.pagemantle_hard import (
    PageMantleHardSession,
    build_pagemantle_hard_suite,
    load_pagemantle_hard_jsonl,
    verify_hard_split_isolation,
)
from dial_skill.benchmarks.deepnote_official import partition_snapshot_into_threads

HARD_DATA_DIR = DRIVE_ROOT / "datasets" / "pagemantle_hard"
EVAL_JSONL = HARD_DATA_DIR / "pm_hard_eval_32.jsonl"
STREAM_JSONL = HARD_DATA_DIR / "pm_hard_stream_24.jsonl"
REBUILD_HARD_DATASET = False

pm_env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
pm_env.ensure_extracted()

if REBUILD_HARD_DATASET or not (EVAL_JSONL.is_file() and STREAM_JSONL.is_file()):
    built = build_pagemantle_hard_suite(env=pm_env, seed=2026, output_dir=HARD_DATA_DIR)
    hard_eval_all = built["pm_hard_eval_32"]
    hard_stream_all = built["pm_hard_stream_24"]
    print(f"Built fresh PageMantle-Hard suite -> {HARD_DATA_DIR}")
else:
    hard_eval_all = load_pagemantle_hard_jsonl(EVAL_JSONL)
    hard_stream_all = load_pagemantle_hard_jsonl(STREAM_JSONL)
    print(f"Loaded persisted PageMantle-Hard suite from {HARD_DATA_DIR}")

qa_report = verify_hard_split_isolation(hard_eval_all, hard_stream_all, strict_counts=True)
print("Split-Isolation & Hardness QA:", json.dumps(qa_report, indent=2))

# Zero-cost smoke test on first eval task & first cross-service silence trap
async def _smoke_hard():
    for sample in (hard_eval_all[0], next(t for t in hard_eval_all if t.trap_type == "cross_service_resolved")):
        async with PageMantleHardSession(hard_task=sample, base_env=pm_env) as sess:
            threads = partition_snapshot_into_threads(sess.initial_context)
            tools = sess.get_openai_tool_schemas(read_only=False, include_finish_task=True)
            print(
                f"[SMOKE OK] {sample.task_id} ({sample.category}, intent={sample.intent}, trap={sample.trap_type}) | "
                f"msgs={len(sess.initial_context['messages'])} threads={len(threads)} tools={len(tools)}"
            )

await _smoke_hard()


## Step 2 — Configure Experiment Scope & Run Pre-Evolution 3-Way Evaluation (`pm_hard_eval_32`)

- **Default `MODE = "full"`**: evaluates all `32` held-out hard tasks (`22` action + `10` hard silence traps) across `B0_direct_hard`, `DIAL_Skill_v1_flat` (ablation), and `DIAL_Skill_v2_per_thread_v0` (ours, pre-evolution).
- **Default `HARD_EVAL_MODELS = ["gpt-4.1-mini", "gpt-4.1"]`** (`gpt-4.1-mini` runs all 3 methods; `gpt-4.1` runs Direct + Per-Thread v0 + Online DB-CSE — set `RUN_FLAT_ABLATION_ON_TERRA = True` to include Flat v1 on `gpt-4.1` too).
- Every episode and judgment is saved atomically to `$DRIVE_ROOT/runs/pagemantle_hard/` for lossless Colab resume.

In [ ]:
# === CELL 4: Configure PageMantle-Hard Scope & Atomic JSONL Helpers ===
MODE = "full"  # "full" (32 held-out + 24 stream) | "pilot" (12 held-out + 8 stream)
SEED_STATIC_FROM_PILOT = True  # Auto-inherit pre-evolution static eval records from pilot/ when running full
HARD_EVAL_MODELS = ["gpt-4.1-mini", "gpt-4.1"]
RUN_FLAT_ABLATION_ON_ALL_MODELS = True
DBCSE_BACKBONE = "gpt-4.1"
PRIMARY_JUDGE_MODEL = "gpt-4.1"
RUN_CROSS_FAMILY_JUDGE = bool(AUTODL_API_KEY)
AUTODL_JUDGE_MODEL = "DeepSeek-V4.1-Flash"
FORCE_RERUN = False

RUN_DIR = DRIVE_ROOT / "runs" / "pagemantle_hard" / MODE
RUN_DIR.mkdir(parents=True, exist_ok=True)
EVAL_EPISODES_PATH = RUN_DIR / "episodes_hard_eval.jsonl"
EVAL_JUDGMENTS_PATH = RUN_DIR / "judgments_hard_eval_primary.jsonl"
EVAL_CROSS_JUDGMENTS_PATH = RUN_DIR / "judgments_hard_eval_cross_family.jsonl"
STREAM_EPISODES_PATH = RUN_DIR / "episodes_hard_stream.jsonl"
STREAM_JUDGMENTS_PATH = RUN_DIR / "judgments_hard_stream.jsonl"
EVOLUTION_LOG_PATH = RUN_DIR / "dbcse_hard_evolution_log.jsonl"
EVOLVED_STORE_PATH = RUN_DIR / "skill_store_hard_evolved.json"
SUMMARY_PATH = RUN_DIR / "summary_pagemantle_hard.json"

if MODE == "pilot":
    eval_tasks = (
        [t for t in hard_eval_all if t.expected_action == "INTERVENE"][:8]
        + [t for t in hard_eval_all if t.expected_action == "SILENT"][:4]
    )
    stream_tasks = [t for t in hard_stream_all if t.wave == 1]
else:
    eval_tasks = list(hard_eval_all)
    stream_tasks = list(hard_stream_all)

def load_jsonl_index(path: Path, key_fn):
    records = {}
    if not path.is_file():
        return records
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        records[key_fn(obj)] = obj
    return records

def append_jsonl_record(path: Path, obj: dict) -> None:
    with path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(obj, ensure_ascii=False) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

# Idempotently seed pre-evolution static eval records from pilot/ -> full/ (never copies DB-CSE post-evolution!)
STATIC_PRE_EVO_METHODS = {
    "B0_direct_hard",
    "DIAL_Skill_v1_flat",
    "DIAL_Skill_v2_per_thread_v0",
}
if MODE == "full" and SEED_STATIC_FROM_PILOT and not FORCE_RERUN:
    pilot_dir = DRIVE_ROOT / "runs" / "pagemantle_hard" / "pilot"
    _seed_key = lambda o: (str(o["method"]), str(o["model_id"]), str(o["task_id"]))
    for fname in (
        "episodes_hard_eval.jsonl",
        "judgments_hard_eval_primary.jsonl",
        "judgments_hard_eval_cross_family.jsonl",
    ):
        src_p = pilot_dir / fname
        dst_p = RUN_DIR / fname
        if not src_p.is_file():
            continue
        existing = load_jsonl_index(dst_p, _seed_key)
        pilot_recs = load_jsonl_index(src_p, _seed_key)
        added = 0
        for k, obj in pilot_recs.items():
            if k[0] in STATIC_PRE_EVO_METHODS and k not in existing:
                append_jsonl_record(dst_p, obj)
                added += 1
        if added > 0:
            print(f"[AUTO-SEED] Inherited {added} pre-evolution static records from pilot/{fname} -> full/{fname}")

print(
    f"MODE={MODE} | Held-out eval={len(eval_tasks)} (silence_traps={sum(1 for t in eval_tasks if t.expected_action == 'SILENT')}) | "
    f"Online DB-CSE stream={len(stream_tasks)} (silence_traps={sum(1 for t in stream_tasks if t.expected_action == 'SILENT')})"
)


In [ ]:
# === CELL 5: Run Pre-Evolution 3-Way Held-Out Eval (Direct vs Flat-v1 vs Per-Thread-v2) ===
from dial_skill.benchmarks.deepnote_official import (
    DeepnoteOfficialJudge,
    DeepnoteRubricJudgment,
    OfficialDialSkillAgent,
    OfficialDialSkillV2Agent,
    OfficialDirectAgent,
    OfficialEpisodeResult,
)
from dial_skill.core.skill_store import SkillStore
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

api_cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache_pagemantle_hard.sqlite")
openai_client = OpenAIClient(api_key=OPENAI_API_KEY, models_cfg=models_cfg, cache=api_cache)
triage_client = TriageClient(
    models_cfg=models_cfg,
    cache=api_cache,
    openai_client=openai_client,
    luna_model="gpt-4.1-mini",
    jev_api_key=JEV_API_KEY,
)
seed_store = SkillStore.from_json_file()
primary_judge = DeepnoteOfficialJudge(
    openai_client=openai_client,
    judge_model=PRIMARY_JUDGE_MODEL,
    num_passes=2,
    seed=42,
)

def _ep_key(obj: dict) -> tuple[str, str, str]:
    return (str(obj["method"]), str(obj["model_id"]), str(obj["task_id"]))

if FORCE_RERUN:
    for p in (EVAL_EPISODES_PATH, EVAL_JUDGMENTS_PATH, EVAL_CROSS_JUDGMENTS_PATH):
        if p.exists():
            p.unlink()

completed_eval_eps = load_jsonl_index(EVAL_EPISODES_PATH, _ep_key)
eval_judgments_raw = load_jsonl_index(EVAL_JUDGMENTS_PATH, _ep_key)
print(f"Resumed eval episodes={len(completed_eval_eps)}, judgments={len(eval_judgments_raw)}")

async def _run_and_grade_eval_suite(method_name: str, stored_mid: str, agent):
    print(f"\n--- Evaluating {method_name} [{stored_mid}] on {len(eval_tasks)} PageMantle-Hard tasks ---")
    for idx, task in enumerate(eval_tasks, start=1):
        key = (method_name, stored_mid, task.task_id)
        if key in completed_eval_eps:
            ep = OfficialEpisodeResult.model_validate(completed_eval_eps[key])
        else:
            async with PageMantleHardSession(hard_task=task, base_env=pm_env) as sess:
                ep = await agent.run_task(sess)
            ep_dict = ep.model_dump(mode="json")
            append_jsonl_record(EVAL_EPISODES_PATH, ep_dict)
            completed_eval_eps[key] = ep_dict

        if key in eval_judgments_raw:
            j = DeepnoteRubricJudgment.model_validate(eval_judgments_raw[key])
        else:
            j = await primary_judge.grade_episode(task=task.to_official_task(), episode=ep)
            j_dict = j.model_dump(mode="json")
            append_jsonl_record(EVAL_JUDGMENTS_PATH, j_dict)
            eval_judgments_raw[key] = j_dict

        if idx % 4 == 0 or idx == len(eval_tasks) or j.overall_score < 6.0:
            print(
                f"  [{idx:02d}/{len(eval_tasks)}] {task.task_id:22s} ({task.intent:7s}) -> "
                f"pred={ep.predicted_action:9s} reads={ep.read_calls_used:2d} overall={j.overall_score:4.1f}"
                f"{' [ZERO]' if j.serious_mistake_zero else ''}"
            )

for model_id in HARD_EVAL_MODELS:
    await _run_and_grade_eval_suite(
        "B0_direct_hard",
        model_id,
        OfficialDirectAgent(
            openai_client=openai_client,
            model=model_id,
            method_name="B0_direct_hard",
            max_turns=16,
        ),
    )
    if RUN_FLAT_ABLATION_ON_ALL_MODELS or model_id == HARD_EVAL_MODELS[0]:
        await _run_and_grade_eval_suite(
            "DIAL_Skill_v1_flat",
            f"{model_id}+{model_id}",
            OfficialDialSkillAgent(
                triage_client=triage_client,
                openai_client=openai_client,
                gate_model=model_id,
                executor_model=model_id,
                skill_store=seed_store,
                cfg=base_cfg,
                method_name="DIAL_Skill_v1_flat",
                recon_max_calls=6,
                recon_max_turns=4,
                stage3_max_turns=14,
            ),
        )
    await _run_and_grade_eval_suite(
        "DIAL_Skill_v2_per_thread_v0",
        f"{model_id}+{model_id}",
        OfficialDialSkillV2Agent(
            triage_client=triage_client,
            openai_client=openai_client,
            gate_model=model_id,
            executor_model=model_id,
            skill_store=seed_store,
            cfg=base_cfg,
            method_name="DIAL_Skill_v2_per_thread_v0",
            recon_max_calls=8,
            recon_max_turns=4,
            stage3_max_turns=16,
        ),
    )

print(f"\nPre-evolution evaluation done! Cumulative OpenAI cost: ${openai_client.usage.total_cost_usd:.4f}")


## Step 3 — Online DB-CSE Evolution on `pm_hard_stream_24` (3 Waves) → Untouched Held-Out Re-Evaluation

Runs our REAL **`DBCSEUpdater`** (`EscrowGate` counterfactual replay via `rescore_slate_with_store`, anti-overfitting proper-noun/ID guardrails, and $O(1)$ prompt capacity cap!) strictly on `pm_hard_stream_24` (zero record-ID overlap with `pm_hard_eval_32`), then evaluates the evolved `SkillStore` (`DIAL_Skill_v2_dbcse`) on the untouched `pm_hard_eval_32` held-out set!

In [ ]:
# === CELL 6: Online DB-CSE Evolution Across 3 Stream Waves + Held-Out Post-Evolution Eval ===
from dial_skill.benchmarks.pagemantle_hard import extract_feedback_from_hard_episode
from dial_skill.evolution.db_cse import DBCSEUpdater
from dial_skill.schemas import SharedEvidenceSlate

if EVOLVED_STORE_PATH.is_file() and not FORCE_RERUN:
    evolved_store = SkillStore.from_json_file(EVOLVED_STORE_PATH)
    dbcse_wave_telemetry = json.loads((RUN_DIR / "dbcse_wave_telemetry.json").read_text(encoding="utf-8"))
    print(f"Loaded persisted evolved SkillStore v{evolved_store.version} from {EVOLVED_STORE_PATH}")
else:
    evolved_store = SkillStore.from_json_file()
    dbcse_updater = DBCSEUpdater(
        skill_store=evolved_store,
        openai_client=openai_client,
        critic_model=DBCSE_BACKBONE,
        cfg=base_cfg,
        log_path=EVOLUTION_LOG_PATH,
        seed=42,
    )
    dbcse_wave_telemetry = {
        "initial_prompt_tokens": evolved_store.estimate_serialized_tokens(),
        "initial_rule_count": evolved_store.total_rule_count(),
        "waves": [],
    }

    for wave_num in sorted({t.wave for t in stream_tasks if t.wave is not None}):
        w_tasks = [t for t in stream_tasks if t.wave == wave_num]
        w_scores = []
        print(f"\n=== Online DB-CSE Wave {wave_num} ({len(w_tasks)} stream tasks, store v{evolved_store.version}) ===")
        for task in w_tasks:
            stream_agent = OfficialDialSkillV2Agent(
                triage_client=triage_client,
                openai_client=openai_client,
                gate_model=DBCSE_BACKBONE,
                executor_model=DBCSE_BACKBONE,
                skill_store=evolved_store,
                cfg=base_cfg,
                method_name=f"DIAL_Skill_v2_stream_w{wave_num}",
                recon_max_calls=8,
                recon_max_turns=4,
                stage3_max_turns=16,
                include_primary_slate_in_metadata=True,
            )
            async with PageMantleHardSession(hard_task=task, base_env=pm_env) as sess:
                ep = await stream_agent.run_task(sess)
            j = await primary_judge.grade_episode(task=task.to_official_task(), episode=ep)
            w_scores.append(j.overall_score)
            append_jsonl_record(STREAM_EPISODES_PATH, ep.model_dump(mode="json"))
            append_jsonl_record(STREAM_JUDGMENTS_PATH, j.model_dump(mode="json"))

            fb = extract_feedback_from_hard_episode(ep, task, j)
            raw_slate = ep.metadata.get("primary_slate")
            if raw_slate is not None and ep.gate is not None:
                slate_obj = SharedEvidenceSlate.model_validate(raw_slate)

                async def _rescore(sl, cand_store, _agent=stream_agent, _skill=fb.skill_name):
                    return await _agent.rescore_slate_with_store(
                        slate=sl, candidate_store=cand_store, focus_skill=_skill
                    )

                evo_entry = await dbcse_updater.process_episode(
                    slate=slate_obj,
                    gate=ep.gate,
                    feedback=fb,
                    rescore_fn=_rescore,
                )
                status_tag = (
                    f"PROMOTED v{evo_entry.version_after}"
                    if evo_entry.promoted
                    else (evo_entry.rejection_reason or fb.feedback_type)
                )
            else:
                status_tag = f"{fb.feedback_type} (no slate)"

            print(
                f"  [W{wave_num}] {task.task_id:24s} | score={j.overall_score:4.1f} | "
                f"fb={fb.feedback_type:20s} | {status_tag}"
            )

        dbcse_wave_telemetry["waves"].append(
            {
                "wave": wave_num,
                "mean_stream_score": round(sum(w_scores) / max(1, len(w_scores)), 4),
                "store_version": evolved_store.version,
                "prompt_tokens": evolved_store.estimate_serialized_tokens(),
                "rule_count": evolved_store.total_rule_count(),
                "stats": dict(dbcse_updater.stats),
            }
        )

    evolved_store.save(EVOLVED_STORE_PATH)
    dbcse_wave_telemetry["final_stats"] = dict(dbcse_updater.stats)
    dbcse_wave_telemetry["token_growth_ratio"] = round(
        evolved_store.estimate_serialized_tokens() / max(1, dbcse_wave_telemetry["initial_prompt_tokens"]), 3
    )
    (RUN_DIR / "dbcse_wave_telemetry.json").write_text(
        json.dumps(dbcse_wave_telemetry, indent=2), encoding="utf-8"
    )
    print("\nSaved evolved SkillStore ->", EVOLVED_STORE_PATH)

print("DB-CSE Evolution Telemetry:", json.dumps(dbcse_wave_telemetry, indent=2))

# Post-Evolution Held-Out Evaluation on UNTOUCHED pm_hard_eval_32!
evolved_agent = OfficialDialSkillV2Agent(
    triage_client=triage_client,
    openai_client=openai_client,
    gate_model=DBCSE_BACKBONE,
    executor_model=DBCSE_BACKBONE,
    skill_store=evolved_store,
    cfg=base_cfg,
    method_name="DIAL_Skill_v2_dbcse",
    recon_max_calls=8,
    recon_max_turns=4,
    stage3_max_turns=16,
)
await _run_and_grade_eval_suite(
    "DIAL_Skill_v2_dbcse",
    f"{DBCSE_BACKBONE}+{DBCSE_BACKBONE}",
    evolved_agent,
)


## Step 4 — Cross-Family Judge Verification, 3-Benchmark Difficulty Calibration & Paper Artifacts

1. Runs optional cross-family grading (`DeepSeek-V4.1-Flash` via AutoDL) on a stratified 24-episode slice of `PageMantle-Hard`.
2. Prints the full **PageMantle-Hard Results Table** (including `Direct` vs `Flat v1` vs `Per-Thread v2` vs `Online DB-CSE`).
3. Prints the **3-Benchmark Difficulty Calibration Table** (`Deepnote Official-8` vs `PageMantle-Hard` vs legacy `PageMantle-Ext`), proving `PageMantle-Hard` matches Official-8's real difficulty profile!
4. Exports standalone `tab_pagemantle_hard.tex` and `fig_pagemantle_hard.pdf`.

In [ ]:
# === CELL 7: Cross-Family Agreement, Full Diagnostics & Standalone Paper Artifacts ===
import matplotlib.pyplot as plt
import numpy as np
from dial_skill.benchmarks.deepnote_official import summarize_official_8_results
from dial_skill.eval.judge import compute_cohens_kappa, compute_spearman_rho

# 1. Optional Cross-Family Judge Agreement (stratified up to 24 episodes to keep runtime fast)
cross_judgments_raw = load_jsonl_index(EVAL_CROSS_JUDGMENTS_PATH, _ep_key)
cross_family_agreement = None
hard_by_id = {t.task_id: t for t in eval_tasks}

if RUN_CROSS_FAMILY_JUDGE and AUTODL_API_KEY:
    autodl_client = OpenAIClient(
        api_key=AUTODL_API_KEY,
        models_cfg=models_cfg,
        cache=api_cache,
        base_url=AUTODL_BASE_URL,
        provider_name="autodl",
    )
    cross_judge = DeepnoteOfficialJudge(
        openai_client=autodl_client,
        judge_model=AUTODL_JUDGE_MODEL,
        num_passes=1,
        seed=101,
    )
    sample_keys = [
        k for k in sorted(completed_eval_eps)
        if k[0] in ("B0_direct_hard", "DIAL_Skill_v2_per_thread_v0", "DIAL_Skill_v2_dbcse")
        and DBCSE_BACKBONE in k[1]
    ][:24]
    for key in sample_keys:
        if key in cross_judgments_raw:
            continue
        ep = OfficialEpisodeResult.model_validate(completed_eval_eps[key])
        task = hard_by_id[ep.task_id]
        try:
            j = await cross_judge.grade_episode(task=task.to_official_task(), episode=ep)
            j_dict = j.model_dump(mode="json")
            append_jsonl_record(EVAL_CROSS_JUDGMENTS_PATH, j_dict)
            cross_judgments_raw[key] = j_dict
        except Exception as exc:
            print(f"  [WARN] Cross-family judge failed on {key}: {type(exc).__name__}: {exc}")

    paired_keys = sorted(set(eval_judgments_raw) & set(cross_judgments_raw))
    if len(paired_keys) >= 6:
        s_a = [float(eval_judgments_raw[k]["overall_score"]) for k in paired_keys]
        s_b = [float(cross_judgments_raw[k]["overall_score"]) for k in paired_keys]
        rho_info = compute_spearman_rho(s_a, s_b)
        b_a = ["zero" if x <= 0.5 else ("mid" if x < 7.0 else "high") for x in s_a]
        b_b = ["zero" if x <= 0.5 else ("mid" if x < 7.0 else "high") for x in s_b]
        cross_family_agreement = {
            "n_paired": len(paired_keys),
            "primary_judge": PRIMARY_JUDGE_MODEL,
            "secondary_judge": AUTODL_JUDGE_MODEL,
            "spearman_rho": rho_info["rho"],
            "spearman_p": rho_info["p_value"],
            "cohens_kappa_3band": compute_cohens_kappa(b_a, b_b),
            "mae": round(sum(abs(x - y) for x, y in zip(s_a, s_b)) / len(s_a), 4),
        }
        print("Cross-Family Judge Agreement (PageMantle-Hard):", json.dumps(cross_family_agreement, indent=2))

# 2. Aggregate Summary Table for PageMantle-Hard
method_specs = []
for m in HARD_EVAL_MODELS:
    method_specs.append((m, "B0_direct_hard", m))
    method_specs.append((m, "DIAL_Skill_v1_flat", f"{m}+{m}"))
    method_specs.append((m, "DIAL_Skill_v2_per_thread_v0", f"{m}+{m}"))
    if m == DBCSE_BACKBONE:
        method_specs.append((m, "DIAL_Skill_v2_dbcse", f"{m}+{m}"))

agg_rows = []
summary_by_group = {}
for backbone, method_name, stored_mid in method_specs:
    eps = [
        OfficialEpisodeResult.model_validate(completed_eval_eps[(method_name, stored_mid, t.task_id)])
        for t in eval_tasks
        if (method_name, stored_mid, t.task_id) in completed_eval_eps
    ]
    if not eps:
        continue
    jmap = {
        t.task_id: DeepnoteRubricJudgment.model_validate(eval_judgments_raw[(method_name, stored_mid, t.task_id)])
        for t in eval_tasks
        if (method_name, stored_mid, t.task_id) in eval_judgments_raw
    }
    stats = summarize_official_8_results(eps, jmap)
    # Also compute Cross-Service Hidden Silence Trap accuracy separately!
    cs_sil_tasks = [t for t in eval_tasks if t.trap_type == "cross_service_resolved"]
    cs_sil_ok = sum(
        1 for t in cs_sil_tasks
        if (method_name, stored_mid, t.task_id) in completed_eval_eps
        and completed_eval_eps[(method_name, stored_mid, t.task_id)]["predicted_action"] == "SILENT"
    )
    stats["cross_service_sil_correct"] = f"{cs_sil_ok}/{len(cs_sil_tasks)}"
    summary_by_group[f"{method_name}__{backbone}"] = stats
    agg_rows.append({"backbone": backbone, "method": method_name, **stats})

agg_df = pd.DataFrame(agg_rows)
print("\n=== 1. PAGEMANTLE-HARD HELD-OUT RESULTS (Strict Deepnote 0-10 Rubric) ===")
display(agg_df)

# 3. Unified 3-Benchmark Difficulty Calibration Table (Official-8 vs PageMantle-Hard vs Legacy PageMantle-Ext)
off8_path = DRIVE_ROOT / "runs" / "deepnote_official_8" / "summary_deepnote_official_8.json"
ext_path = DRIVE_ROOT / "runs" / "main_pagemantle" / "full" / "pre_evolution_metrics.json"
off8_agg = json.loads(off8_path.read_text(encoding="utf-8")).get("aggregate", {}) if off8_path.is_file() else {}
ext_agg = json.loads(ext_path.read_text(encoding="utf-8")) if ext_path.is_file() else {}

calibration_rows = []
for backbone, ext_d_key, ext_s_key in [
    ("gpt-4.1-mini", "B0_direct_luna", "DIAL_Skill_v0_luna"),
    ("gpt-4.1", "B0_direct_terra", "DIAL_Skill_v0_terra"),
]:
    for label, off_k, hard_k, ext_k in [
        ("Direct", f"B0_official_direct__{backbone}", f"B0_direct_hard__{backbone}", ext_d_key),
        ("DIAL-Skill (Per-Thread v2)", f"DIAL_Skill_v2_thread_isolated__{backbone}", f"DIAL_Skill_v2_per_thread_v0__{backbone}", ext_s_key),
    ]:
        calibration_rows.append(
            {
                "backbone": backbone,
                "method": label,
                "Official_8": off8_agg.get(off_k, {}).get("overall_mean"),
                "PageMantle_Hard": summary_by_group.get(hard_k, {}).get("overall_mean"),
                "Legacy_PageMantle_Ext": ext_agg.get(ext_k, {}).get("overall_score"),
                "Hard_SilAcc": summary_by_group.get(hard_k, {}).get("silence_correct"),
                "Hard_CrossSvc_Sil": summary_by_group.get(hard_k, {}).get("cross_service_sil_correct"),
            }
        )

cal_df = pd.DataFrame(calibration_rows)
print("\n=== 2. 3-BENCHMARK DIFFICULTY CALIBRATION (Official-8 vs PageMantle-Hard vs PageMantle-Ext) ===")
display(cal_df)

# 4. Save Summary JSON, LaTeX Table, and Publication PDF Figure
summary_payload = {
    "benchmark": "pagemantle_hard_v1",
    "mode": MODE,
    "qa_report": qa_report,
    "eval_models": HARD_EVAL_MODELS,
    "dbcse_backbone": DBCSE_BACKBONE,
    "dbcse_wave_telemetry": dbcse_wave_telemetry,
    "cross_family_agreement": cross_family_agreement,
    "aggregate": summary_by_group,
    "calibration_vs_official8_and_ext": calibration_rows,
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

tex_lines = [
    "% Auto-generated by Notebook 09 (PageMantle-Hard Benchmark + Online DB-CSE)",
    "\\begin{table}[t]",
    "\\centering",
    "\\caption{Held-out evaluation on \\textsc{PageMantle-Hard} (32 multi-channel tasks with 2 decoy threads/task, 6 cross-service hidden silence traps, mixed-intent Guard, and strict Deepnote \\texttt{important\\_mistakes} zero-condition rubric).}",
    "\\label{tab:pagemantle_hard}",
    "\\small",
    "\\begin{tabular}{llccccccc}",
    "\\toprule",
    "Backbone & Method & Overall & Int/Sil & Content & SilAcc & Cross-Svc Sil & ActRec & Avg Reads \\\\",
    "\\midrule",
]
_m_lbl = {
    "B0_direct_hard": "Direct (Official Prompt)",
    "DIAL_Skill_v1_flat": "DIAL-Skill (Flat Snapshot)",
    "DIAL_Skill_v2_per_thread_v0": "DIAL-Skill (Per-Thread v0)",
    "DIAL_Skill_v2_dbcse": "\\textbf{DIAL-Skill + Online DB-CSE}",
}
for r in agg_rows:
    ce_s = f"{r['correctness_and_evidence_mean']:.2f}" if r["correctness_and_evidence_mean"] is not None else "--"
    tex_lines.append(
        f"{r['backbone']} & {_m_lbl.get(r['method'], r['method'])} & {r['overall_mean']:.2f} & "
        f"{r['intervention_or_silence_mean']:.2f} & {ce_s} & {r['silence_correct']} & "
        f"{r['cross_service_sil_correct']} & {r['action_correct']} & {r['avg_read_calls']:.1f} \\\\"
    )
tex_lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
tex_path = DRIVE_ROOT / "paper_artifacts" / "tables" / "tab_pagemantle_hard.tex"
tex_path.write_text("\n".join(tex_lines) + "\n", encoding="utf-8")
print("Saved LaTeX table:", tex_path)

fig, axes = plt.subplots(1, 2, figsize=(14.0, 4.9), dpi=200)
# Panel A: PageMantle-Hard Held-Out Progression on DBCSE_BACKBONE
focus_methods = [
    ("B0_direct_hard", "Direct", "#9CA3AF"),
    ("DIAL_Skill_v1_flat", "DIAL (Flat v1)", "#60A5FA"),
    ("DIAL_Skill_v2_per_thread_v0", "DIAL (Per-Thread v0)", "#2563EB"),
    ("DIAL_Skill_v2_dbcse", "DIAL + DB-CSE", "#0F766E"),
]
labels_a, vals_a, sils_a, cols_a = [], [], [], []
for m_k, m_l, col in focus_methods:
    st = summary_by_group.get(f"{m_k}__{DBCSE_BACKBONE}")
    if st:
        labels_a.append(m_l)
        vals_a.append(float(st["overall_mean"]))
        sils_a.append(10.0 * float(st["silence_accuracy"] or 0.0))
        cols_a.append(col)
xa = np.arange(len(labels_a))
w = 0.36
b1 = axes[0].bar(xa - w / 2, vals_a, width=w, label="Overall Score (0-10)", color=cols_a)
b2 = axes[0].bar(xa + w / 2, sils_a, width=w, label="Silence Acc (x10)", color="#F59E0B", alpha=0.82)
for b, v in zip(b1, vals_a):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 0.15, f"{v:.2f}", ha="center", va="bottom", fontsize=8.5)
axes[0].set_xticks(xa)
axes[0].set_xticklabels(labels_a, fontsize=8.5)
axes[0].set_ylim(0, 11.0)
axes[0].set_ylabel("Score (0-10)")
axes[0].set_title(f"(a) PageMantle-Hard Held-Out Progression ({DBCSE_BACKBONE})")
axes[0].grid(axis="y", linestyle="--", alpha=0.35)
axes[0].legend(frameon=False, fontsize=8.5)

# Panel B: 3-Benchmark Difficulty Calibration
xb = np.arange(len(cal_df))
wb = 0.25
axes[1].bar(xb - wb, cal_df["Legacy_PageMantle_Ext"].fillna(0.0), width=wb, label="Legacy PageMantle-Ext", color="#CBD5E1")
axes[1].bar(xb, cal_df["PageMantle_Hard"].fillna(0.0), width=wb, label="PageMantle-Hard (Ours)", color="#2563EB")
axes[1].bar(xb + wb, cal_df["Official_8"].fillna(0.0), width=wb, label="Deepnote Official-8", color="#DC2626")
axes[1].set_xticks(xb)
axes[1].set_xticklabels(
    [f"{'DIAL' if 'DIAL' in r.method else 'Direct'}\n({r.backbone})" for r in cal_df.itertuples()],
    fontsize=8.5,
)
axes[1].set_ylim(0, 11.0)
axes[1].set_ylabel("Mean Score (0-10)")
axes[1].set_title("(b) Benchmark Difficulty Calibration: Hard vs Official-8 vs Ext")
axes[1].grid(axis="y", linestyle="--", alpha=0.35)
axes[1].legend(frameon=False, fontsize=8.5)

plt.tight_layout()
fig_path = DRIVE_ROOT / "paper_artifacts" / "figures" / "fig_pagemantle_hard.pdf"
fig.savefig(fig_path, bbox_inches="tight")
plt.show()
print("Saved figure:", fig_path)


In [ ]:
# === CELL 8: Optional Auto-Disconnect Colab Runtime (Saves Compute Units) ===
AUTO_DISCONNECT_COLAB = False  # Set True for unattended runs

import time

if IN_COLAB and AUTO_DISCONNECT_COLAB:
    print("Flushing Drive buffers and disconnecting Colab runtime in 5 seconds...")
    os.sync()
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Notebook 09 complete! Runtime left active (set AUTO_DISCONNECT_COLAB=True to auto-stop).")
